In [ ]:
from typing import Dict, List
import json5
import pathlib
current_dir = pathlib.Path().resolve()
print('eval directory',current_dir)
print('parent directory',current_dir.parent)


In [ ]:
# Create a .env file in eval directory 
# Set env variables BEDROCK_AWS_ACCESS_KEY_ID, BEDROCK_AWS_SECRET_ACCESS_KEY,BEDROCK_AWS_REGION
import dotenv
dotenv.load_dotenv()

In [ ]:
urns_dict: Dict[str, List[str]] = json5.loads(
    (current_dir / "eval_urns.json").read_text()
)
print(urns_dict.keys())

In [ ]:
# Copy graph_credentials.json file to parent docs_generation directory
# generated using https://github.com/acryldata/experimental/blob/main/hsheth/bulk-graph-creds/generate_many_graph_credentials.py 

for deployment, urn_list in urns_dict.items():
    if deployment == "acryl":
        continue # facing some issues with acryl token
    for urn_obj in urns_list:
        urn = urn_obj["urn"]
        
        

In [13]:
updated_prompt_template = '''\
You are tasked with generating concise descriptions for a DataHub table and its columns based on provided metadata. Here is the information you will be working with:

<table_info>
{table_info}
</table_info>

<column_info>
{column_info}
</column_info>

Generate the descriptions as follows:

1. Table Description:
   Create a few paragraphs of Markdown-formatted text that includes:
   a) A summary of the primary purpose and business importance of the table.
   b) If metadata is available, a summary of the upstream tables and transformations applied.
   c) A summary of the downstream tables (consumers) and general use cases for the table. Only include information that can be substantiated by the provided table_info.
   d) Technical notes and usage tips, including the table type (fact or dimension) and grain if available.
   e) A note on whether the table directly contains any PII data, like names, emails, and addresses. Do not provide recommendations related to access control, monitoring, or governance.

   Format any references to other entities as markdown links, using the entity URN as the link. For example: [table_name](urn:li:dataset:(urn:li:dataPlatform:snowflake,database.schema.table_name,PROD))
   Use Markdown sections like H2 and H3 with appropriate section titles. The first line should be "# <table name>", followed by a blank line.

2. Column Descriptions:
   For each column, create a concise description of one or two sentences. Prefer elliptical sentences that are direct and to the point. If available, include details about how the column was generated or calculated.

When writing the descriptions:
- Aim for a technical yet informative tone, suitable for a data catalog.
- Avoid weak phrases like "suggests", "could be", "likely", or "is considered". Only include information you are confident about based on the provided metadata.
- Be concise and to the point.

Provide your output in the following dictionary format:

{{
    "table_description": """
[Your multi-line table description here]
""",
    "column_name1": "Column description",
    "column_name2": "Column description",
    ...
}}

Ensure that the dictionary is properly formatted and parsable. Use the column display names as keys for the column descriptions. Include the table description with the key "table_description".\
'''